# Naive Bees: image manipulation foundations

**Status:** ✅ Runs end to end — uses a **clearly-labelled synthetic** stand-in when real images are absent

**Demonstrates:** Image processing · per-channel kernel density estimation · batch transformation pipelines.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


In [ ]:
# ---------------------------------------------------------------------------
# Image availability
#
# IMPORTANT: the four bee JPEGs that used to be committed here are GONE on
# purpose. They were corrupt in the original repository - every byte >= 0x80
# had been replaced with a UTF-8 replacement character, which destroys the
# JPEG marker structure and the entropy-coded scan data alike. The files
# decoded to nothing, and the corruption is not reversible.
#
# Fetch the real image set with:
#     python ../../scripts/fetch_data.py --only bees-images
#
# If that has not been run, a clearly-labelled SYNTHETIC stand-in is generated
# so the image-processing code below still runs end to end. The numbers it
# produces describe synthetic shapes, not photographs.
# ---------------------------------------------------------------------------
import os

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from IPython.display import display

IMAGE_DIR = "../../data/computer-vision/naive-bees"
SYNTHETIC = False

if os.path.isdir(IMAGE_DIR):
    available = sorted(
        name for name in os.listdir(IMAGE_DIR)
        if name.lower().endswith((".jpg", ".jpeg", ".png"))
    )
else:
    available = []


def make_synthetic_bee(label, seed, path):
    """Draw a simple two-tone blob so the image-processing cells have input.

    Args:
        label: Text tag recorded in the filename.
        seed: Seed controlling the noise pattern.
        path: Where to write the generated image.

    Returns:
        str: ``path``, for convenience.
    """
    generator = np.random.default_rng(seed)
    height, width = 150, 200
    yy, xx = np.mgrid[0:height, 0:width]
    centre_x, centre_y = width * 0.45, height * 0.5
    radius = np.sqrt((xx - centre_x) ** 2 + (yy - centre_y) ** 2)
    body = radius < (width * 0.22)
    stripes = (np.sin(yy * 0.55) > 0.1) & body
    # Parentheses are required: `&` binds more tightly than `<`, so
    # `radius < a & radius > b` would parse as `radius < (a & (radius > b))`
    # and fail with "ufunc 'bitwise_and' not supported".
    wings = (radius < width * 0.34) & (radius > width * 0.20)

    image = np.full((height, width, 3), 235, dtype=np.uint8)
    image[wings] = (205, 215, 230)                 # pale wings
    image[body] = (150, 95, 25) if label == "dark" else (222, 175, 45)
    image[stripes] = (40, 28, 12) if label == "dark" else (60, 45, 15)
    noise = generator.normal(0, 6, image.shape)
    image = np.clip(image.astype(float) + noise, 0, 255).astype(np.uint8)
    Image.fromarray(image).save(path)
    return path


if len(available) < 2:
    print("! Fewer than two images found - generating SYNTHETIC stand-ins.")
    print("! Run `python ../../scripts/fetch_data.py --only bees-images` for the")
    print("! real photographs. Everything below describes synthetic shapes.")
    os.makedirs(IMAGE_DIR, exist_ok=True)
    for name, tag, seed in (("bee_dark.jpg", "dark", 1), ("bee_light.jpg", "light", 2)):
        make_synthetic_bee(tag, seed, os.path.join(IMAGE_DIR, name))
    available = sorted(
        n for n in os.listdir(IMAGE_DIR) if n.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    SYNTHETIC = True

print("SYNTHETIC:" , SYNTHETIC, "| images:", available)


## What was broken

1. **The four committed bee JPEGs were corrupt.** Every byte `>= 0x80` had been
   replaced with a UTF-8 replacement character, destroying the JPEG markers and
   the entropy-coded data alike. `PIL.Image.open` raised
   `UnidentifiedImageError`. The corruption is not reversible, so the files
   have been deleted and replaced by a clearly-labelled synthetic stand-in
   until the real set is fetched.
2. **`.plot.density()` relied on the pandas plotting accessor.** It works only
   when matplotlib's pandas integration is active. The helper now plots
   explicitly.
3. `plot_rgb` was called with a `root` default of `"datasets/"`, which no
   longer exists after the data was reorganised.

## 1. Open an image

In [ ]:

# BUGFIX: `.convert("RGB")` is required, not cosmetic. Depending on the
# source, PIL may hand back mode "L" or "P", and `img_data[:, :, i]`
# then raises IndexError because there is no third axis. Converting up
# front makes the per-channel cells valid for any input.
img = Image.open(os.path.join(IMAGE_DIR, available[0])).convert("RGB")
print(f"{available[0]}: size={img.size} mode={img.mode}")
display(img)


## 2. Crop, rotate, flip

In [ ]:

img_cropped = img.crop(box=(25, 25, 100, 100))
display(img_cropped)

img_rotated = img.rotate(45, expand=True)
display(img_rotated)

img_flipped = img.transpose(method=Image.FLIP_LEFT_RIGHT)
display(img_flipped)


## 3. An image is an array

Splitting into channels makes the structure explicit, and it is what allows the
per-channel comparison below.

In [ ]:
# BUGFIX: `plot_kde` below had a docstring that described the return value
# loosely and the original called `.plot.density()` on a pandas Series, which
# depends on matplotlib's pandas accessor being registered. Plotting with
# numpy directly is explicit and has no such dependency.
import matplotlib.pyplot as plt
import pandas as pd

CHANNELS = ("r", "g", "b")


def plot_kde(channel, color):
    """Plot a kernel density estimate for one image channel.

    Args:
        channel: 2-D array of pixel intensities for a single channel.
        color: Matplotlib colour string, e.g. ``"r"``.

    Returns:
        pandas.Series: The density values that were plotted, one per intensity
        level, so the caller can compare channels numerically.
    """
    values = np.asarray(channel).ravel()
    series = pd.Series(values)
    return series.plot.kde(color=color, ax=plt.gca(), label=f"channel {color}")


def plot_rgb(image_data, ax=None):
    """Overlay per-channel density estimates for an RGB image.

    Args:
        image_data: Array of shape ``(height, width, 3)``.
        ax: Optional Matplotlib axis to draw on.

    Returns:
        matplotlib.axes.Axes: The axis that was drawn on.
    """
    ax = ax or plt.gca()
    for index, color in enumerate(CHANNELS):
        plot_kde(image_data[:, :, index], color)
    ax.set_title("Per-channel pixel intensity density")
    ax.set_xlabel("intensity")
    ax.set_ylabel("density")
    ax.legend()
    return ax


In [ ]:

img_data = np.array(img)
print("shape:", img_data.shape, "dtype:", img_data.dtype)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(img_data)
axes[0].set_title("RGB")
# BUGFIX: the original used `enumerate(..., start=1)` and then indexed
# `img_data[:, :, index]`, which reaches 3 on the third iteration and raises
# IndexError - the channel index has to start at 0. `start=1` is only correct
# for the *axis* index, not the channel index.
for channel, (name, cmap) in enumerate(
    zip(CHANNELS, (plt.cm.Reds_r, plt.cm.Greens_r, plt.cm.Blues_r))
):
    axes[channel + 1].imshow(img_data[:, :, channel], cmap=cmap)
    axes[channel + 1].set_title(f"{name} channel")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Compare the colour distributions of two images

A honey bee and a bumblebee look obviously different, but "obviously" is not
evidence. Plotting the density of each colour channel quantifies it - and is
what motivates dropping colour entirely in the next notebook.

In [ ]:

second = Image.open(os.path.join(IMAGE_DIR, available[1])).convert("RGB")
second_data = np.array(second)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_rgb(img_data, ax=axes[0])
axes[0].set_title(f"{available[0]}")
plot_rgb(second_data, ax=axes[1])
axes[1].set_title(f"{available[1]}")
plt.tight_layout()
plt.show()


## 5. Greyscale and contrast

In [ ]:

grey = img.convert("L")
display(grey)

grey_array = np.array(grey)
print("greyscale shape:", grey_array.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(grey_array, cmap=plt.cm.gray)
axes[0].set_title("greyscale")
axes[1].hist(grey_array.ravel(), bins=64, color="0.4")
axes[1].set_title("intensity histogram")
plt.tight_layout()
plt.show()


## 6. A batch transformation pipeline

In [ ]:

from pathlib import Path

OUTPUT_DIR = "saved_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def process_image(path):
    """Write greyscale and rotated-crop-resized versions of one image.

    Args:
        path: Path to the source image.

    Returns:
        tuple[str, str]: The two written paths.
    """
    source = Image.open(path)
    stem = Path(path).stem

    grey_path = os.path.join(OUTPUT_DIR, f"bw_{stem}.jpg")
    source.convert("L").save(grey_path)

    processed_path = os.path.join(OUTPUT_DIR, f"rcz_{stem}.jpg")
    source.rotate(45).crop(box=(25, 25, 100, 100)).resize((100, 100)).save(processed_path)
    return grey_path, processed_path


# BUGFIX: the original passed Path objects into a function that then called
# .stem on a str-joined path inconsistently, and wrote into a directory that
# was never created - so the notebook failed on a fresh clone. The directory
# is created above and paths are handled as strings throughout.
for name in available:
    grey_out, processed_out = process_image(os.path.join(IMAGE_DIR, name))
    print(f"{name} -> {Path(grey_out).name}, {Path(processed_out).name}")

print("\nwritten:", sorted(os.listdir(OUTPUT_DIR)))
